# Calcul de l'enveloppe convexe du système Fe-Cu-Co-Ni 
Données extraites de Materials Project

In [1]:
import numpy as np
import pandas as pd
from convexhull import cvhull
import time

## Donnée d'entrée
* name : liste d'identificateur des composé (doit être différent pour chaque composé)
* composition : tableau des composition (ou nombre d'atome)
* energy : liste des énergie (Hf, E_DFT,...)

In [2]:
df = pd.read_csv("data/incomplet.csv")
df

,nom,A,B,C,D,somme,energie
0,Fe,1,0,0,0,1,-8.19
1,Co,0,0,1,0,1,-7.11
2,Ni,0,0,0,1,1,-5.37
3,CuNi,0,1,0,1,2,-4.91
4,FeCuNi,1,1,0,1,3,-6.05
5,CoNi,0,0,1,1,2,-7.00
6,CoNi2,0,0,1,1,2,-6.50


In [3]:
name = df.index
composition = df.loc[:,['A','B','C','D']]
energy = df['energie']
cvhull(name , composition, energy)

ValueError: All reference (purs elements) must be in database

Attention, une seul contrainte toute les référence doivent être présent dans la base

In [4]:
df = pd.read_csv("data/data.csv")
df

,id,formule,Fe,Cu,Co,Ni,somme,E
0,mp-1245108,Fe,1,0,0,0,1,-8.19
1,mp-1271198,Fe,1,0,0,0,1,-8.42
2,mp-13,Fe,1,0,0,0,1,-8.47
3,mp-1194030,Fe,1,0,0,0,1,-8.29
4,mp-1271068,Fe,1,0,0,0,1,-8.45
...,...,...,...,...,...,...,...,...
78,mp-1225698,Cu₃Ni,0,3,0,1,4,-4.47
79,mp-1225695,CuNi,0,1,0,1,2,-4.87
80,mp-1225687,CuNi,0,1,0,1,2,-4.92
81,mp-1184069,CuNi,0,1,0,1,2,-4.91


In [11]:
energy = df.loc[:,'E']

name = [x.strip() for x in df.loc[:,'id']]

element = ['Fe','Cu','Co','Ni']
composition = df.loc[:,element]

energy.shape, len(name), composition.shape

((83,), 83, (83, 4))

In [12]:
start = time.time()

hull = cvhull(name, composition, energy)

end = time.time()

print(f"Temps d'exécution : {end - start:.10f} secondes")

Temps d'exécution : 0.0099465847 secondes


In [13]:
hull.stable

array([ 2, 13, 21, 29, 35, 38, 40, 55, 58, 68])

## Liste des phases stable

In [14]:
hull.print_stables()

The database consist of 83 compounds with 10 stable

List of stable compounds:
-------------------------
-----------------------------------------------------------
  i   Name               A       B       C       D   Energy
-----------------------------------------------------------
  2   mp-13         1.0000  0.0000  0.0000  0.0000    -8.47
 13   mp-54         0.0000  0.0000  1.0000  0.0000    -7.11
 21   mp-23         0.0000  0.0000  0.0000  1.0000    -5.78
 29   mp-30         0.0000  1.0000  0.0000  0.0000    -4.10
 35   mp-18695      0.9375  0.0000  0.0625  0.0000    -8.40
 38   mp-601820     0.7500  0.0000  0.2500  0.0000    -8.18
 40   mp-2090       0.5000  0.0000  0.5000  0.0000    -7.86
 55   mp-1418       0.2500  0.0000  0.0000  0.7500    -6.54
 58   mp-2213       0.5000  0.0000  0.0000  0.5000    -7.19
 68   mp-1183837    0.0000  0.0000  0.7500  0.2500    -6.80
-----------------------------------------------------------


## Information pour un composé donné

### Energie

In [7]:
float(hull.energy_compound('mp-2090'))

-7.86

### Décomposition

In [10]:
dH = hull.distance_hull('mp-1225687')

The compound mp-1225687 is not stable
--------------------------------------------
ΔH = 0.02
Decomposition:  0.5(mp-23) + 0.5(mp-30) 


### Info complete

In [8]:
dh = hull.info_compond('mp-2090')

Info about compound : mp-2090)
------------------------------------
E = -7.86
The compound mp-2090 is stable!


In [9]:
hull.info_compond('mp-1006883')

Info about compound : mp-1006883)
------------------------------------
E = -6.29
The compound mp-1006883 is not stable
--------------------------------------------
ΔH = 0.17
Decomposition:  0.6667(mp-1183837) + 0.3333(mp-23) 


In [11]:
dH = hull.distance_hull('mp-1225687', verbose=False)
print(f"DfH = {dH:.2f}")

DfH = 0.02


## Information pour une composition donnée

In [16]:
compo = [0.2,0.5,0.2,0.1]
hull.equilibrium(compo)
print(f"Hf = {hull.energie_hull(compo):.2f}")

Decomposition:  0.5(mp-30) + 0.34(mp-2090) + 0.12(mp-1418) + 0.04(mp-1183837) 
Hf = -5.78


## Ecart à l'enveloppe pour tous les composés

In [17]:
dic = hull.table_dhull()
pd.DataFrame(dic)

,Name,A,B,C,D,Stability,Deviation from hull
0,mp-1245108,1.000000,0.000000,0.0,0.000000,False,0.28
1,mp-1271198,1.000000,0.000000,0.0,0.000000,False,0.05
2,mp-13,1.000000,0.000000,0.0,0.000000,True,0.00
3,mp-1194030,1.000000,0.000000,0.0,0.000000,False,0.18
4,mp-1271068,1.000000,0.000000,0.0,0.000000,False,0.02
...,...,...,...,...,...,...,...
78,mp-1225698,0.000000,0.750000,0.0,0.250000,False,0.05
79,mp-1225695,0.000000,0.500000,0.0,0.500000,False,0.07
80,mp-1225687,0.000000,0.500000,0.0,0.500000,False,0.02
81,mp-1184069,0.000000,0.500000,0.0,0.500000,False,0.03
